# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [30]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
# Bỏ mọi module src.* đã nạp: reload riêng augment không nạp lại extract/view/landmarks nó import,
# nên kernel chạy lâu sẽ giữ bản cũ (vd. ImportError PITCH_AUG_DEG).
for name in [m for m in sys.modules if m == 'src' or m.startswith('src.')]: del sys.modules[name]
reader = importlib.import_module('src.data.data_reader')
augment = importlib.import_module('src.data.augment')
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'  # 'all', 'xin chào', hoặc ['xin chào', 'cảm ơn']
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))  # N; -1 = tăng cường toàn bộ, không giới hạn
COPIES_PER_SAMPLE = 2    # chỉ dùng khi MIN_SAMPLES_PER_CLASS = -1
NOISE_STRENGTH = 1.0    # khoảng (0, 3]
SEED = None  # đặt số nguyên để chạy tái lập
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')


95 mẫu dự kiến tạo trong 27 lớp


## Kiểm tra trước khi tạo


In [31]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


a: hiện có 130, thêm 0, sau đó có 130
age: hiện có 130, thêm 0, sau đó có 130
b: hiện có 130, thêm 0, sau đó có 130
cảm ơn: hiện có 130, thêm 0, sau đó có 130
dấu huyền: hiện có 130, thêm 0, sau đó có 130
dấu nặng: hiện có 130, thêm 0, sau đó có 130
dấu sắc: hiện có 130, thêm 0, sau đó có 130
dấu â,ô,ê: hiện có 130, thêm 0, sau đó có 130
dấu ơ,ư: hiện có 130, thêm 0, sau đó có 130
e: hiện có 130, thêm 0, sau đó có 130
g: hiện có 130, thêm 0, sau đó có 130
h: hiện có 130, thêm 0, sau đó có 130
i: hiện có 130, thêm 0, sau đó có 130
k: hiện có 130, thêm 0, sau đó có 130
l: hiện có 130, thêm 0, sau đó có 130
m: hiện có 130, thêm 0, sau đó có 130
n: hiện có 130, thêm 0, sau đó có 130
o: hiện có 130, thêm 0, sau đó có 130
r: hiện có 130, thêm 0, sau đó có 130
t: hiện có 130, thêm 0, sau đó có 130
ten_la: hiện có 130, thêm 0, sau đó có 130
toi: hiện có 5, thêm 95, sau đó có 100
u: hiện có 130, thêm 0, sau đó có 130
v: hiện có 130, thêm 0, sau đó có 130
xin chào: hiện có 130, thêm 0, sau đó có

## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [32]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


[1/5] toi/0 -> 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23
[2/5] toi/1 -> 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42
[3/5] toi/2 -> 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61
[4/5] toi/3 -> 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80
[5/5] toi/4 -> 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
Đã tạo 95 mẫu; lỗi 0


## Kiểm tra group


In [33]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 3480
Tổng: {'a': 130, 'age': 130, 'b': 130, 'cảm ơn': 130, 'dấu huyền': 130, 'dấu nặng': 130, 'dấu sắc': 130, 'dấu â,ô,ê': 130, 'dấu ơ,ư': 130, 'e': 130, 'g': 130, 'h': 130, 'i': 130, 'k': 130, 'l': 130, 'm': 130, 'n': 130, 'o': 130, 'r': 130, 't': 130, 'ten_la': 130, 'toi': 100, 'u': 130, 'v': 130, 'xin chào': 130, 'y': 130, 'địa chỉ': 130}
Group gốc độc lập: {'a': 5, 'age': 15, 'b': 5, 'cảm ơn': 15, 'dấu huyền': 5, 'dấu nặng': 15, 'dấu sắc': 5, 'dấu â,ô,ê': 5, 'dấu ơ,ư': 5, 'e': 5, 'g': 15, 'h': 30, 'i': 15, 'k': 15, 'l': 5, 'm': 30, 'n': 15, 'o': 5, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 5, 'u': 15, 'v': 5, 'xin chào': 15, 'y': 16, 'địa chỉ': 15}
3154 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
